In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
# SOURCE AND TARGET PATHS

drivers_source_path = f"{s3_root_path}/Silver/Aaisha/drivers"
ref_nationality_region_path = f"{s3_root_path}/Gold/Aaisha/ref_nationality_region"

target_path = f"{s3_root_path}/Gold/Aaisha/dim_drivers"

print("Drivers Source    :", drivers_source_path)
print("Reference Source  :", ref_nationality_region_path)
print("Target Path       :", target_path)

In [0]:
drivers_df = (
    spark.read
        .format("delta")
        .load(drivers_source_path)
)

ref_nationality_region_df = (
    spark.read
        .format("delta")
        .load(ref_nationality_region_path)
)

In [0]:
dim_drivers_df = (
    drivers_df
        .join(
            ref_nationality_region_df,
            drivers_df.nationality == ref_nationality_region_df.nationality,
            "left"
        )
        .select(
            drivers_df.driverId.alias("driver_id"),
            F.concat_ws(
                " ",
                drivers_df.givenName,
                drivers_df.familyName
            ).alias("driver_name"),
            drivers_df.dateOfBirth.alias("date_of_birth"),
            drivers_df.nationality,
            ref_nationality_region_df.region.alias("nationality_region")
        )
)

display(dim_drivers_df)

In [0]:
(
    dim_drivers_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

In [0]:
gold_drivers_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(gold_drivers_df)